In [1]:
!pip install dash==3.2.0 dash-bootstrap-components pandas plotly

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.0/204.0 kB 7.9 MB/s eta 0:00:00


In [2]:
import pandas as pd
import plotly.graph_objects as go
import dash
from dash import Dash, dcc, html, dash_table, Input, Output, State, ctx
import dash_bootstrap_components as dbc
from datetime import date
print('Dependencies loaded successfully.')

Dependencies loaded successfully.


In [3]:
crayfish_css = """
    body { background-color: #f3f6f9; font-family: Arial, Helvetica, sans-serif; color: #1e293b; font-size: 0.95rem; }
    .card { background-color: #ffffff; border: 1px solid #e2e8f0; border-radius: 8px; box-shadow: 0 4px 12px rgba(0, 0, 0, 0.02); }
    .brand-title { color: #1a446c; font-size: 1.4rem; }
    .section-header { font-size: 0.95rem; font-weight: 700; color: #334155; border-bottom: 1px solid #f1f5f9; padding-bottom: 6px; margin-bottom: 12px; }
    .text-navy { color: #1a446c !important; }
    .text-danger { color: #dc3545 !important; }
    .text-success { color: #198754 !important; }
"""

def kpi_card(title, value, color_hex):
    return dbc.Card([
        dbc.CardBody([
            html.Div(title, className="text-muted fw-bold mb-2", style={'fontSize': '11px', 'textTransform': 'uppercase', 'letterSpacing': '0.5px'}),
            html.H3(value, className="fw-bold mb-0", style={'color': color_hex})
        ], className="text-center p-2")
    ], className="shadow-sm h-100 border-0", style={'borderRadius': '8px'})

def form_input(label, id_val, req=False):
    return html.Div([
        html.Label(f"{label} {'*' if req else ''}", className="fw-semibold text-secondary", style={'fontSize': '12px'}),
        dbc.Input(type="number", id=id_val, min=0, step="any", className="mb-2 shadow-none")
    ])

print('Styling functions ready!')

Styling functions ready!


In [4]:
# Farmers Data / client
baseline_logs = [
    {'date': '2026-01-10', 'freq': 'Monthly', 'stocked': 100, 'harvested': 0, 'deaths': 5, 'feed': 2, 'gain': 0, 'females': 20, 'berried': 0, 'eggs': 0, 'larvae': 0, 'juv': 0, 'revenue': 0, 'expenses': 5000},
    {'date': '2026-03-15', 'freq': 'Monthly', 'stocked': 0, 'harvested': 0, 'deaths': 2, 'feed': 10, 'gain': 3, 'females': 0, 'berried': 5, 'eggs': 200, 'larvae': 800, 'juv': 0, 'revenue': 0, 'expenses': 1500},
    {'date': '2026-06-01', 'freq': 'Monthly', 'stocked': 0, 'harvested': 40, 'deaths': 1, 'feed': 15, 'gain': 5, 'females': 15, 'berried': 8, 'eggs': 250, 'larvae': 1500, 'juv': 400, 'revenue': 6000, 'expenses': 1800}
]

NAVY = "#1a446c"
SUCCESS = "#198754"
DANGER = "#dc3545"

app = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

# Dashboard Layout
app.layout = html.Div(style={'backgroundColor': '#f3f6f9', 'fontFamily': 'Arial, sans-serif', 'color': '#1e293b', 'minHeight': '100vh', 'padding': '20px'}, children=[
    dcc.Store(id='memory-store', data=baseline_logs),

    dbc.Container([
        dbc.Row([

            dbc.Col(md=3, children=[
                dbc.Card([
                    dbc.CardBody([
                        html.H4("Crayfish N' Friends", className="fw-bold", style={'color': NAVY, 'fontSize': '1.4rem'}),
                        html.P("Jovy Pagapulan Garcia / Sibalom Antique", className="text-muted small border-bottom pb-3"),

                        html.H6("Log Period", className="fw-bold mt-3", style={'color': '#334155'}),
                        dbc.Row([
                            dbc.Col([html.Label("Date", className="fw-semibold text-secondary", style={'fontSize':'12px'}), dbc.Input(type="date", id="in-date", value=date.today().strftime('%Y-%m-%d'))], width=6),
                            dbc.Col([html.Label("Freq", className="fw-semibold text-secondary", style={'fontSize':'12px'}), dbc.Select(id="in-freq", options=[{"label": "Daily", "value": "Daily"}, {"label": "Monthly", "value": "Monthly"}], value="Daily")], width=6)
                        ], className="mb-3"),

                        html.H6("Production & Growth", className="fw-bold mt-2", style={'color': '#334155'}),
                        dbc.Row([dbc.Col(form_input("Stock Added", "in-stocked"), width=6), dbc.Col(form_input("Harvested", "in-harvested"), width=6)]),
                        dbc.Row([dbc.Col(form_input("Feed(kg)", "in-feed"), width=4), dbc.Col(form_input("Gain(kg)", "in-gain"), width=4), dbc.Col(form_input("Deaths", "in-deaths"), width=4)]),

                        html.H6("Hatchery / Breeding", className="fw-bold mt-2", style={'color': '#334155'}),
                        dbc.Row([dbc.Col(form_input("Females", "in-females"), width=6), dbc.Col(form_input("Berried", "in-berried"), width=6)]),
                        dbc.Row([dbc.Col(form_input("Eggs", "in-eggs"), width=4), dbc.Col(form_input("Larvae", "in-larvae"), width=4), dbc.Col(form_input("Juv Harv", "in-juv"), width=4)]),

                        html.H6("Financials (PHP)", className="fw-bold mt-2", style={'color': '#334155'}),
                        dbc.Row([dbc.Col(form_input("Revenue", "in-revenue"), width=6), dbc.Col(form_input("Expenses", "in-expenses"), width=6)]),

                        dbc.Button("Save Entry", id="btn-save", color="primary", className="w-100 fw-bold mt-3 mb-2"),
                        dbc.Button("Clear All Data", id="btn-clear", color="light", className="w-100 text-danger border-danger fw-semibold")
                    ])
                ], className="h-100 border-0 shadow-sm", style={'borderRadius': '8px'})
            ]),


            dbc.Col(md=9, children=[
                dbc.Row(id='kpi-row', className="mb-3"),
                dbc.Row([
                    dbc.Col(dbc.Card(dcc.Graph(id="chart-pie", style={'height': '320px'}), className="p-2 shadow-sm border-0", style={'borderRadius': '8px'}), md=4),
                    dbc.Col(dbc.Card(dcc.Graph(id="chart-line", style={'height': '320px'}), className="p-2 shadow-sm border-0", style={'borderRadius': '8px'}), md=8),
                ], className="mb-3"),
                dbc.Row([
                    dbc.Col(dbc.Card(dcc.Graph(id="chart-fcr", style={'height': '320px'}), className="p-2 shadow-sm border-0", style={'borderRadius': '8px'}), md=6),
                    dbc.Col(dbc.Card(dcc.Graph(id="chart-breeding", style={'height': '320px'}), className="p-2 shadow-sm border-0", style={'borderRadius': '8px'}), md=6),
                ], className="mb-3"),

                dbc.Card([
                    html.H5("Historical Logs", className="fw-bold p-3 mb-0", style={'color': NAVY}),
                    dash_table.DataTable(
                        id='data-table',
                        style_header={'backgroundColor': '#1a252f', 'color': 'white', 'fontWeight': 'bold'},
                        style_cell={'textAlign': 'left', 'padding': '10px', 'fontFamily': 'Arial'},
                        style_data_conditional=[{'if': {'row_index': 'odd'}, 'backgroundColor': '#f8f9fa'}]
                    )
                ], className="shadow-sm border-0 overflow-hidden", style={'borderRadius': '8px'})
            ])
        ])
    ], fluid=True, className="py-3")
])

# Dashboard logicc
@app.callback(
    Output('memory-store', 'data'),
    Output('kpi-row', 'children'),
    Output('chart-pie', 'figure'), Output('chart-line', 'figure'),
    Output('chart-fcr', 'figure'), Output('chart-breeding', 'figure'), Output('data-table', 'data'),
    Input('btn-save', 'n_clicks'), Input('btn-clear', 'n_clicks'),
    State('memory-store', 'data'), State('in-date', 'value'), State('in-freq', 'value'),
    State('in-stocked', 'value'), State('in-harvested', 'value'), State('in-feed', 'value'), State('in-gain', 'value'), State('in-deaths', 'value'),
    State('in-females', 'value'), State('in-berried', 'value'), State('in-eggs', 'value'), State('in-larvae', 'value'), State('in-juv', 'value'),
    State('in-revenue', 'value'), State('in-expenses', 'value')
)
def update_dashboard(save_clicks, clear_clicks, data, date_val, freq, stocked, harvested, feed, gain, deaths, females, berried, eggs, larvae, juv, revenue, expenses):
    trigger = ctx.triggered_id
    if trigger == 'btn-clear':
        data = baseline_logs.copy()
    elif trigger == 'btn-save':
        s_base = float(stocked or 0)
        s_juv = float(juv or 0)
        new_entry = {
            'date': date_val, 'freq': freq, 'stocked': s_base + s_juv, 'harvested': float(harvested or 0),
            'feed': float(feed or 0), 'gain': float(gain or 0), 'deaths': float(deaths or 0),
            'females': float(females or 0), 'berried': float(berried or 0), 'eggs': float(eggs or 0),
            'larvae': float(larvae or 0), 'juv': s_juv, 'revenue': float(revenue or 0), 'expenses': float(expenses or 0)
        }
        data.append(new_entry)
        data = sorted(data, key=lambda x: x['date'])

    df = pd.DataFrame(data)

    if df.empty:
        empty_fig = go.Figure().update_layout(template='simple_white')
        return data, [], empty_fig, empty_fig, empty_fig, empty_fig, []

    # Math logic
    t_stocked = df['stocked'].sum()
    t_harvested = df['harvested'].sum()
    t_deaths = df['deaths'].sum()
    t_juv = df['juv'].sum()

    m_surv = (t_harvested / t_stocked * 100) if t_stocked > 0 else 0
    m_fcr = (df['feed'].sum() / df['gain'].sum()) if df['gain'].sum() > 0 else 0
    m_margin = ((df['revenue'].sum() - df['expenses'].sum()) / df['revenue'].sum() * 100) if df['revenue'].sum() > 0 else 0
    m_mortality = (t_deaths / t_stocked * 100) if t_stocked > 0 else 0
    m_unaccounted = max(0, 100 - (m_surv + m_mortality))

    # KPIs Row Generation
    kpis = [
        dbc.Col(kpi_card("Total Stocked", f"{t_stocked:,.0f}", NAVY), width=2),
        dbc.Col(kpi_card("Total Deaths", f"{t_deaths:,.0f}", DANGER), width=2),
        dbc.Col(kpi_card("Juv Harvested", f"{t_juv:,.0f}", SUCCESS), width=2),
        dbc.Col(kpi_card("Avg Survival", f"{m_surv:.1f}%", SUCCESS if m_surv >= 80 else DANGER), width=2),
        dbc.Col(kpi_card("Avg FCR", f"{m_fcr:.2f}", SUCCESS if (m_fcr > 0 and m_fcr <= 2.0) else DANGER), width=2),
        dbc.Col(kpi_card("Profit Margin", f"{m_margin:.1f}%", SUCCESS if m_margin > 0 else DANGER), width=2),
    ]


    df['fcr'] = df.apply(lambda r: (r['feed'] / r['gain']) if r['gain'] > 0 else 0, axis=1)
    df['margin'] = df.apply(lambda r: ((r['revenue'] - r['expenses']) / r['revenue'] * 100) if r['revenue'] > 0 else 0, axis=1)
    df['b_succ'] = df.apply(lambda r: (r['berried'] / r['females'] * 100) if r['females'] > 0 else 0, axis=1)
    df['h_rate'] = df.apply(lambda r: (r['larvae'] / (r['berried'] * r['eggs']) * 100) if (r['berried'] * r['eggs']) > 0 else 0, axis=1)
    df['j_surv'] = df.apply(lambda r: (r['juv'] / r['larvae'] * 100) if r['larvae'] > 0 else 0, axis=1)
    df['date_lbl'] = df['date'].astype(str) + ' (' + df['freq'].astype(str).str[0] + ')'

    # Mortality Pie Chart
    pie = go.Figure(go.Pie(labels=['Survival Rate', 'Mortality Rate', 'Unaccounted / Missing'], values=[m_surv, m_mortality, m_unaccounted], hole=0.5, marker_colors=['#198754', '#dc3545', '#cbd5e1'], textinfo='none'))
    pie.update_layout(title=dict(text="Overall Stock Mortality Dynamic", font=dict(size=14, color=NAVY)), margin=dict(t=40, b=20, l=20, r=20), plot_bgcolor='white', paper_bgcolor='white', legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5))

    # Financial Line Chart
    line = go.Figure()
    line.add_trace(go.Scatter(x=df['date_lbl'], y=df['revenue'], name='Revenue', mode='lines+markers', fill='tozeroy', fillcolor='rgba(13, 110, 253, 0.05)', line=dict(color='#0d6efd', width=2)))
    line.add_trace(go.Scatter(x=df['date_lbl'], y=df['expenses'], name='Expenses', mode='lines+markers', line=dict(color='#ffc107', width=2)))
    line.update_layout(title=dict(text="Financial Trend Over Time", font=dict(size=14, color=NAVY)), margin=dict(t=40, b=30, l=20, r=20), hovermode="x unified", plot_bgcolor='white', paper_bgcolor='white', yaxis=dict(gridcolor='#f1f5f9'), legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5))

    # FCR Bar Chart
    bar_fcr = go.Figure(go.Bar(x=df['date_lbl'], y=df['fcr'], name='FCR History', marker_color='#6f42c1', width=0.4))
    bar_fcr.update_layout(title=dict(text="Feed Conversion Ratio History", font=dict(size=14, color=NAVY)), margin=dict(t=40, b=30, l=20, r=20), plot_bgcolor='white', paper_bgcolor='white', yaxis=dict(gridcolor='#f1f5f9'), legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5))

    # Hatchery Combo Chart
    bar_hatch = go.Figure()
    bar_hatch.add_trace(go.Bar(x=df['date_lbl'], y=df['b_succ'], name='Breeding Success %', marker_color='rgba(13, 110, 253, 0.7)', width=0.3))
    bar_hatch.add_trace(go.Bar(x=df['date_lbl'], y=df['h_rate'], name='Egg Hatch Rate %', marker_color='rgba(255, 193, 7, 0.7)', width=0.3))
    bar_hatch.add_trace(go.Scatter(x=df['date_lbl'], y=df['j_surv'], name='Juv Survival Rate %', mode='lines+markers', line=dict(color='#20c997', width=2)))
    bar_hatch.update_layout(barmode='group', title=dict(text="Hatchery Metrics & Efficiency Breakdown", font=dict(size=14, color=NAVY)), margin=dict(t=40, b=30, l=20, r=20), hovermode="x unified", plot_bgcolor='white', paper_bgcolor='white', yaxis=dict(gridcolor='#f1f5f9', range=[0, 100]), legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5))

    # Data Table Processing
    tbl = df[['date', 'freq', 'stocked', 'harvested', 'deaths', 'juv', 'fcr', 'revenue', 'margin']].copy()
    tbl.columns = ['Date', 'Freq', 'Stock Added', 'Harvested', 'Deaths', 'Juv Harvested', 'FCR', 'Revenue (PHP)', 'Margin']
    tbl['FCR'] = tbl['FCR'].apply(lambda x: f"{x:.2f}")
    tbl['Revenue (PHP)'] = tbl['Revenue (PHP)'].apply(lambda x: f"₱{x:,.0f}")
    tbl['Margin'] = tbl['Margin'].apply(lambda x: f"{x:.1f}%")

    return data, kpis, pie, line, bar_fcr, bar_hatch, tbl.to_dict('records')

if __name__ == '__main__':
    app.run(debug=True, jupyter_mode="inline", jupyter_height=1500)

<IPython.core.display.Javascript object>